# Tiny Siamese — SECOND: HPO v3 → seeds → final → benchmark

Порядок основан на Heavy_Full_HPO_v3_metadata_fix.ipynb. После обновления — Restart Kernel.
Абсолютные бюджеты [40,60,80]: HPO 40, seed validation 60, final training 80.
40→60 = ещё 20 эпох. Final: seeds 42,123,777, без early stopping. Test отдельно
через run_final_benchmark. По умолчанию данные и реальные запуски отключены.
GT ROI baseline измеряет семантику известных регионов, а не качество всей detector-системы.


In [ ]:
import json
import sys
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/satchange").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside satellite-change-intelligence.")
if not (3, 11) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError("Use the project Python 3.11/3.12 kernel.")
sys.path.insert(0, str(ROOT / "src"))
print("Repository:", ROOT, "Interpreter:", sys.executable)

Все изменяемые пути и флаги — здесь. `LOCAL_HOLDOUT=True` явно разрешает фиксированный local
holdout из train, если размеченного val/test нет. Это не официальный test и не доказанная
географическая независимость. `SPLIT_SEED` не зависит от training seeds.
Для scalar indexed labels подтвердите документированную mapping локальной копии через
`USE_DOCUMENTED_INDEXED_IDS`; RGB/palette PNG проверяются автоматически.

In [ ]:
DATA_ROOT = ROOT / "data/SECOND"
CACHE_ROOT = ROOT / "data/SECOND_semantic_cache"
CONFIG_FILE = ROOT / "configs/tiny_siamese_change_classifier.yaml"
LOCAL_ARCHIVE = None  # Path to a local official ZIP, or use the installed DATA_ROOT.
ALLOW_DOWNLOAD = False
PREPARE_DATA = False
LOCAL_HOLDOUT = True
SPLIT_SEED = 2099
USE_DOCUMENTED_INDEXED_IDS = False
SOURCE_GROUPS = None  # Optional mapping sample_id -> verified source scene group.
SENSOR_METADATA = None  # Only metadata actually available from your source.
INSTALL_MISSING = False  # Installs small missing notebook/data packages; never torch.

GPU_IDS = [0, 1, 2, 3]  # Visible numbering under the existing CUDA_VISIBLE_DEVICES.
HPO_ROOT = ROOT / "artifacts/hyperparameter_search/semantic/tiny_siamese_second"
SEARCH_NAME = "second_oracle_hpo_v3_40_60_80"
SEARCH_DIRECTORY = HPO_ROOT / SEARCH_NAME
STAGE_EPOCHS = [40, 60, 80]  # ABSOLUTE HPO / seeds / final targets.
FINAL_SEEDS = [42, 123, 777]
DEPLOYMENT_SEED = 42  # Fixed before test.
REUSE_DIRECTORIES = []  # Own compatible Tiny searches only.
CHECKPOINTS = []  # Own complete state checkpoints; automatic recovery prefers last.pt.
ALLOW_LEGACY_IMPORT = False
RESTORE_SAVED_PROTOCOL = True
SEED_QUEUE_POLICY = "all"  # Fresh search: all. finish_started requires started secondary seeds.
SELECTION_GAP = 0.04
PATIENCE = None
MIN_EPOCHS = 1
MIN_DELTA = 0.0
RUN_HPO_BLOCKS = False
PLAN_SEEDS = False
RUN_SEEDS = False
PLAN_FINAL = False
RUN_FINAL_TRAINING = False
RUN_TEST_BENCHMARK = False
RUN_PIPELINE_EXAMPLE = False

CLASSIFIER_CHECKPOINT = None  # None -> selected deployment best.pt once summary exists.
DETECTOR_CHECKPOINT = None
DETECTOR_INPUT_SIZE = (256, 256)  # Match this detector's actual whole-image preprocessing.
DETECTOR_OUTPUT_KIND = "logits"
DETECTOR_PRECISION = None  # Use saved detector precision; explicit override is recorded.
EXAMPLE_BEFORE = None
EXAMPLE_AFTER = None
EXAMPLE_MAP = None
EXAMPLE_MASK_SOURCE = "ground_truth"  # Or explicit predicted with DETECTOR_IDENTITY.
DETECTOR_IDENTITY = None
EVALUATION_PRECISION = None  # None reproduces the saved numerical mode.

In [ ]:
import importlib.util
import subprocess

requirements = {
    "numpy": "numpy",
    "PIL": "pillow",
    "cv2": "opencv-python",
    "yaml": "PyYAML",
    "gdown": "gdown",
    "matplotlib": "matplotlib",
}
missing = [
    package for module, package in requirements.items() if importlib.util.find_spec(module) is None
]
if missing and INSTALL_MISSING:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
elif missing:
    raise RuntimeError(
        f"Missing dependencies {missing}; select INSTALL_MISSING in the config cell."
    )

In [ ]:
import torch

import satchange
from satchange.semantic.data import SemanticROIDataset, read_map, read_rgb
from satchange.semantic.second import bootstrap_second, discover_second, prepare_second
from satchange.utils import load_config

print("Package:", satchange.__file__, "Torch:", torch.__version__)
print("Visible CUDA GPUs:", torch.cuda.device_count())
CONFIG = load_config(CONFIG_FILE)
protocol_path = SEARCH_DIRECTORY / "pipeline_protocol.json"
SAVED_PROTOCOL = (
    json.loads(protocol_path.read_text())
    if RESTORE_SAVED_PROTOCOL and protocol_path.is_file()
    else None
)
if SAVED_PROTOCOL:
    CONFIG.update(SAVED_PROTOCOL["base"]["semantic_config"])
    CONFIG["dropout"] = SAVED_PROTOCOL["base"]["model_parameters"]["dropout"]
    for key in (
        "seed",
        "batch_size",
        "num_workers",
        "learning_rate",
        "weight_decay",
        "scheduler",
        "class_weights",
        "precision",
        "tf32",
    ):
        CONFIG["training"][key] = SAVED_PROTOCOL["base"][key]
print("Saved protocol loaded:", SAVED_PROTOCOL is not None)
if SAVED_PROTOCOL and (
    SAVED_PROTOCOL["context"].get("task") != "semantic_roi_classification"
    or SAVED_PROTOCOL["context"]["model_name"] != "tiny_siamese_change_classifier"
):
    raise ValueError("Saved search is not a Tiny semantic experiment.")

Проверка SECOND выполняется до запуска очереди. При установленном комплекте bootstrap пропускает
скачивание. Download использует ссылку страницы [авторов SECOND](https://captain-whu.github.io/SCD/);
quota/login ошибки требуют доступного локального ZIP. Local SHA256 — наша provenance, не checksum авторов.

In [ ]:
PREPARED = None
if PREPARE_DATA:
    installed = bootstrap_second(DATA_ROOT, archive=LOCAL_ARCHIVE, allow_download=ALLOW_DOWNLOAD)
    layout = discover_second(installed)
    print("Labeled scenes by supplied split:", {k: len(v) for k, v in layout.items()})
    PREPARED = prepare_second(
        installed,
        CACHE_ROOT,
        CONFIG,
        split_seed=SPLIT_SEED,
        local_holdout=LOCAL_HOLDOUT,
        indexed_ids=CONFIG["dataset"]["indexed_id_names"] if USE_DOCUMENTED_INDEXED_IDS else None,
        source_groups=SOURCE_GROUPS,
        sensor_metadata=SENSOR_METADATA,
    )
    print("Verified prepared cache:", PREPARED)
elif SAVED_PROTOCOL and Path(SAVED_PROTOCOL["dataset_root"]).is_file():
    PREPARED = Path(SAVED_PROTOCOL["dataset_root"])
    SemanticROIDataset(PREPARED, "train", CONFIG).verify_sources()
else:
    print("Preparation disabled. Enable PREPARE_DATA after choosing the source.")

In [ ]:
if PREPARED is not None:
    document = json.loads(PREPARED.read_text())
    print(json.dumps(document["coverage"], indent=2))
    print("Split provenance:", document.get("second", {}).get("local_splits"))
    print("ROI sizes [H,W] are native; visual display alone may scale them.")
    import matplotlib.pyplot as plt

    dataset = SemanticROIDataset(PREPARED, "train", CONFIG)
    for index in range(min(3, len(dataset))):
        sample = dataset[index]
        fig, axes = plt.subplots(1, 3, figsize=(10, 3))
        for ax, key in zip(axes, ("before", "after", "mask"), strict=True):
            values = (
                sample[key].permute(1, 2, 0).numpy() if key != "mask" else sample[key][0].numpy()
            )
            ax.imshow(values, cmap="gray" if key == "mask" else None)
            ax.set_title(key)
            ax.axis("off")
        fig.suptitle(
            f"{sample['sample_id']}: {CONFIG['classes'][sample['before_label']]} -> "
            f"{CONFIG['classes'][sample['after_label']]}"
        )
        plt.show()
else:
    print("No prepared cache selected; examples skipped.")

Безопасный implementation smoke: разные rectangular/odd ROI без resize/padding, shared
backbone `32→64→128`, feature grid примерно `H/4 × W/4`, маска уменьшается только для pooling,
raw logits `[B,C]`, две CE. Logical batch хранится списком, logits объединяются перед loss.
BatchNorm обновляется на temporal pair, не на всём logical minibatch. Это ограничение baseline.

In [ ]:
from satchange.models import TinySiameseChangeClassifier, count_model_parameters
from satchange.semantic.training import logical_forward, semantic_loss

previous_threads = torch.get_num_threads()
torch.set_num_threads(1)
try:
    torch.manual_seed(42)
    model = TinySiameseChangeClassifier(CONFIG["classes"], CONFIG["dropout"])
    samples = [
        {
            "before": torch.rand(3, h, w),
            "after": torch.rand(3, h, w),
            "mask": torch.ones(1, h, w),
            "before_label": 0,
            "after_label": 1,
        }
        for h, w in [(17, 23), (19, 29)]
    ]
    logits, before_labels, after_labels = logical_forward(model, samples, torch.device("cpu"))
    loss, _ = semantic_loss(logits, before_labels, after_labels)
    loss.backward()
    assert torch.isfinite(loss) and all(
        p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters()
    )
    print("Synthetic shapes:", {k: tuple(v.shape) for k, v in logits.items()})
    print("Finite synthetic CE:", float(loss.detach()), count_model_parameters(model))
finally:
    torch.set_num_threads(previous_threads)
    del model

## Общие параметры v3

Native ROI → shared CNN → две группы logits [B,C] → две CE. Отбор: среднее macro F1
двух дат по всем объявленным классам. Batch/LR → LR → weight decay/dropout → scheduler.
BCE/Dice, image_size и threshold grid здесь не используются. Один worker на GPU;
пул четырёх GPU не добавляет четвёртый seed. Старые Heavy weights несовместимы.
После Restart читается сохранённая конфигурация и активные абсолютные бюджеты.


In [ ]:
import inspect

from satchange.automated_hpo import run_automated_parameter_search
from satchange.semantic.training import hpo_parameters
from satchange.training import fit_binary_change_model

if "stage_epochs" not in inspect.signature(run_automated_parameter_search).parameters:
    raise RuntimeError("Install merged HPO v3 source and Restart Kernel.")
if not {"task", "stop_file"}.issubset(inspect.signature(fit_binary_change_model).parameters):
    raise RuntimeError("Shared trainer lacks Tiny/v3 support; Restart Kernel.")
STAGES = [
    ("01_batch_lr", {"batch_size": [4, 8], "learning_rate": [0.0005, 0.001]}),
    ("02_lr", {"learning_rate": [0.00075, 0.001]}),
    ("03_wd_dropout", {"weight_decay": [0.0001, 0.001], "model__dropout": [0.0, 0.1]}),
    ("04_scheduler", {"scheduler": ["none", "reduce_on_plateau"]}),
]
BASE = hpo_parameters(CONFIG, PREPARED) if PREPARED is not None else None
if SAVED_PROTOCOL:
    STAGES, BASE = SAVED_PROTOCOL["stages"], SAVED_PROTOCOL["base"]
    FINAL_SEEDS = SAVED_PROTOCOL["seeds"]
    SELECTION_GAP = SAVED_PROTOCOL["gaps"]["selection_score"]
    REUSE_DIRECTORIES = SAVED_PROTOCOL["reuse_from"]
    CHECKPOINTS = SAVED_PROTOCOL["initial_checkpoints"]
    target_file = SEARCH_DIRECTORY / "active_phase_targets.json"
    targets = json.loads(target_file.read_text()) if target_file.is_file() else {}
    STAGE_EPOCHS = [
        targets.get(phase, count)
        for phase, count in zip(("hpo", "seeds", "final"), STAGE_EPOCHS, strict=True)
    ]
OPTIONS = dict(
    task="semantic_roi_classification",
    stages=STAGES,
    base_parameters=BASE,
    model_name="tiny_siamese_change_classifier",
    dataset_root=PREPARED,
    stage_epochs=STAGE_EPOCHS,
    seeds=FINAL_SEEDS,
    additional_seeds=True,
    output_root=HPO_ROOT,
    search_name=SEARCH_NAME,
    gpu_ids=GPU_IDS,
    reuse_from=REUSE_DIRECTORIES,
    resume_from=CHECKPOINTS,
    resume_incomplete=True,
    allow_legacy_import=ALLOW_LEGACY_IMPORT,
    patience=PATIENCE,
    min_epochs=MIN_EPOCHS,
    min_delta=MIN_DELTA,
    seed_patience=None,
    selection_gap=SELECTION_GAP,
    joint_tiebreaker=False,
    checkpoint_interval=10,
    max_trial_memory_mb=80_000,
    oom_retry_delay_seconds=30,
    max_oom_attempts=12,
)
if (
    any((RUN_HPO_BLOCKS, PLAN_SEEDS, RUN_SEEDS, PLAN_FINAL, RUN_FINAL_TRAINING, RUN_TEST_BENCHMARK))
    and PREPARED is None
):
    raise FileNotFoundError("Prepare real SECOND first; no synthetic fallback.")
print("HPO / seeds / final targets:", STAGE_EPOCHS, "Final seeds:", FINAL_SEEDS)
print("Prepared:", PREPARED, "Search:", SEARCH_DIRECTORY)

## 01_batch_lr

Включите RUN_HPO_BLOCKS и выполните блок. Предыдущие блоки должны быть завершены.


In [ ]:
if RUN_HPO_BLOCKS:
    hpo_result = run_automated_parameter_search(**OPTIONS, stage="hpo", hpo_stage="01_batch_lr")
    if not hpo_result["complete"]:
        raise RuntimeError("HPO paused; repeat this block before later phases.")
    print("Survivors:", len(hpo_result["survivors"]))
else:
    print("01_batch_lr disabled.")

## 02_lr

Включите RUN_HPO_BLOCKS и выполните блок. Предыдущие блоки должны быть завершены.


In [ ]:
if RUN_HPO_BLOCKS:
    hpo_result = run_automated_parameter_search(**OPTIONS, stage="hpo", hpo_stage="02_lr")
    if not hpo_result["complete"]:
        raise RuntimeError("HPO paused; repeat this block before later phases.")
    print("Survivors:", len(hpo_result["survivors"]))
else:
    print("02_lr disabled.")

## 03_wd_dropout

Включите RUN_HPO_BLOCKS и выполните блок. Предыдущие блоки должны быть завершены.


In [ ]:
if RUN_HPO_BLOCKS:
    hpo_result = run_automated_parameter_search(**OPTIONS, stage="hpo", hpo_stage="03_wd_dropout")
    if not hpo_result["complete"]:
        raise RuntimeError("HPO paused; repeat this block before later phases.")
    print("Survivors:", len(hpo_result["survivors"]))
else:
    print("03_wd_dropout disabled.")

## 04_scheduler

Включите RUN_HPO_BLOCKS и выполните блок. Предыдущие блоки должны быть завершены.


In [ ]:
if RUN_HPO_BLOCKS:
    hpo_result = run_automated_parameter_search(**OPTIONS, stage="hpo", hpo_stage="04_scheduler")
    if not hpo_result["complete"]:
        raise RuntimeError("HPO paused; repeat this block before later phases.")
    print("Survivors:", len(hpo_result["survivors"]))
else:
    print("04_scheduler disabled.")

## План seed-фазы — без обучения

Включите PLAN_SEEDS и проверьте fresh/resume/reuse. Новый поиск требует policy all.
finish_started продолжает уже начатые вторичные seeds; неполные семьи не ранжируются.
dry_run записывает фиксированный план на диск.


In [ ]:
if PLAN_SEEDS:
    seed_plan = run_automated_parameter_search(
        **OPTIONS,
        stage="seeds",
        pending_policy=SEED_QUEUE_POLICY,
        dry_run=True,
    )
    for item in seed_plan["jobs"]:
        saved = item["source"]["epoch"] if item["source"] else 0
        action = "reuse" if item["reused"] else "resume" if saved else "fresh"
        print(item["parameters"]["seed"], action, saved, "->", item["target_epoch"])
    print("Plan:", seed_plan["plan_path"], "Skipped:", seed_plan["skipped_trials"])
else:
    print("Seed plan disabled.")

## Seed validation — target=60

Сравниваются только полные группы с одинаковыми seeds/бюджетом/параметрами.
Победитель выбирается по validation mean per-date macro F1. Test не используется.


In [ ]:
if RUN_SEEDS:
    seed_summary = run_automated_parameter_search(
        **OPTIONS,
        stage="seeds",
        pending_policy=SEED_QUEUE_POLICY,
    )
    if not seed_summary["complete"]:
        raise RuntimeError("Seed phase paused; repeat this cell before final training.")
else:
    path = SEARCH_DIRECTORY / "seed_validation_summary.json"
    seed_summary = json.loads(path.read_text()) if path.is_file() else None
if seed_summary:
    for rank, row in enumerate(seed_summary["ranking"], 1):
        print(rank, row["seeds"], row["mean_selection_score"], row["std_selection_score"])
    print("Incomplete families:", len(seed_summary["incomplete_families"]))
else:
    print("Seed validation summary absent.")

## Final — одна конфигурация, 80 эпох × 3 seeds

final_choice.json фиксируется до обучения и test. Каждый seed продолжает своё состояние
либо начинает с нуля. Early stopping отключён. Из другого kernel можно вызвать
request_hpo_pause(SEARCH_DIRECTORY); для продолжения повторите прежний вызов.
Target никогда не прибавляется сам.


In [ ]:
if PLAN_FINAL:
    final_plan = run_automated_parameter_search(**OPTIONS, stage="final", dry_run=True)
    assert len(final_plan["jobs"]) == 3
    assert {job["parameters"]["seed"] for job in final_plan["jobs"]} == set(FINAL_SEEDS)
    for item in final_plan["jobs"]:
        saved = item["source"]["epoch"] if item["source"] else 0
        print(f"seed={item['parameters']['seed']}: {saved} -> {item['target_epoch']}")
    print("Final plan:", final_plan["plan_path"])
else:
    print("Final plan disabled.")

In [ ]:
if RUN_FINAL_TRAINING:
    final_summary = run_automated_parameter_search(**OPTIONS, stage="final")
    if not final_summary["complete"]:
        raise RuntimeError("Final paused; repeat this cell before benchmark.")
else:
    path = SEARCH_DIRECTORY / "final_training_summary.json"
    final_summary = json.loads(path.read_text()) if path.is_file() else None
checkpoint = Path(CLASSIFIER_CHECKPOINT) if CLASSIFIER_CHECKPOINT else None
if final_summary:
    checkpoints = {
        int(seed): Path(path) for seed, path in final_summary["best_checkpoints"].items()
    }
    if set(checkpoints) != set(FINAL_SEEDS):
        raise RuntimeError("Incomplete final seed set.")
    if any(row["epoch"] != STAGE_EPOCHS[2] for row in final_summary["results"]):
        raise RuntimeError("Final results have a different target budget.")
    checkpoint = checkpoint or checkpoints[DEPLOYMENT_SEED]
    print("Final checkpoints:", checkpoints)
else:
    print("Final training disabled; no final summary selected.")

## run_final_benchmark — validation → test → скорость → export

Включите RUN_TEST_BENCHMARK после окончательного выбора и трёх завершённых final runs.
Сводка перечитывается с диска после Restart. best.pt может быть раньше 80-й эпохи,
если там был лучший validation. Строгую проверку validation сохраняйте включённой.
Сохраняются две confusion matrices, per-class metrics, macro F1 двух дат, joint accuracy,
coverage/provenance и sample std. Скорость измеряет classifier на первом native validation
ROI, без detector, extraction, I/O и device transfer; размеры ROI влияют на latency.


In [ ]:
if RUN_TEST_BENCHMARK:
    from satchange.final_benchmark import run_final_benchmark

    summary_path = SEARCH_DIRECTORY / "final_training_summary.json"
    final_summary = json.loads(summary_path.read_text())
    checkpoints = {
        int(seed): Path(path) for seed, path in final_summary["best_checkpoints"].items()
    }
    if not final_summary["complete"] or set(checkpoints) != set(FINAL_SEEDS):
        raise RuntimeError("Benchmark requires all three completed final seeds.")
    if any(row["epoch"] != STAGE_EPOCHS[2] for row in final_summary["results"]):
        raise RuntimeError("Final target differs from notebook settings.")
    benchmark = run_final_benchmark(
        task="semantic_roi_classification",
        variant="tiny_siamese_second_oracle",
        checkpoints=checkpoints,
        project_root=ROOT,
        dataset_root=PREPARED,
        training_results=summary_path,
        deployment_seed=DEPLOYMENT_SEED,
        device=f"cuda:{GPU_IDS[0]}",
        evaluation_batch_size=1,
        num_workers=0,
        precision=EVALUATION_PRECISION or OPTIONS["base_parameters"]["precision"],
        warmup=20,
        repetitions=100,
        strict_validation=True,
        overwrite=False,
    )
    print("Final artifacts:", benchmark["output_dir"])
else:
    print("Final test disabled; enable RUN_TEST_BENCHMARK after final training.")

Pipeline example: original RGB → frozen detector → aligned probabilities → native crops.
Если detector checkpoint отсутствует, явно выбранная saved/GT map демонстрирует только вторую
часть pipeline. Detector preprocessing здесь поддерживает RGB/255 и whole-image resize;
для иных нормализаций, tiles/padding предоставьте map с известным обратным mapping.

In [ ]:
from satchange.semantic.inference import predict_checkpoint, predict_detector_checkpoint

if RUN_PIPELINE_EXAMPLE:
    if checkpoint is None or EXAMPLE_BEFORE is None or EXAMPLE_AFTER is None:
        raise ValueError("Choose classifier checkpoint and aligned native RGB paths.")
    before, after = read_rgb(EXAMPLE_BEFORE), read_rgb(EXAMPLE_AFTER)
    if DETECTOR_CHECKPOINT is not None:
        prediction = predict_detector_checkpoint(
            checkpoint,
            DETECTOR_CHECKPOINT,
            before,
            after,
            input_size=DETECTOR_INPUT_SIZE,
            output_kind=DETECTOR_OUTPUT_KIND,
            device=f"cuda:{GPU_IDS[0]}" if torch.cuda.is_available() else "cpu",
            detector_precision=DETECTOR_PRECISION,
            classifier_precision=EVALUATION_PRECISION,
        )
    else:
        if EXAMPLE_MAP is None:
            raise ValueError("Choose aligned saved map; no detector is fabricated.")
        prediction = predict_checkpoint(
            checkpoint,
            before,
            after,
            read_map(EXAMPLE_MAP).unsqueeze(0),
            inference_mask_source=EXAMPLE_MASK_SOURCE,
            detector={"identity": DETECTOR_IDENTITY} if DETECTOR_IDENTITY else None,
            precision=EVALUATION_PRECISION,
        )
    print(json.dumps(prediction, indent=2))
else:
    print("Pipeline example disabled.")